# Предобработка данных

In [1]:
import pandas as pd
import numpy as np
import sklearn as sk

In [2]:
data = pd.read_csv('data/adult.csv')

In [ ]:
data.info()

## 1. Преобразования данных

Заменим значения колонки таргета '<=50k' и '>50k', на бинарные значения 0 и 1:

In [ ]:
data['income'] = data['income'].map({'<=50K': 0, '>50K': 1})

Как мы узнали из EDA пропуски выглядят, как строка '?'. Заполним все пропуски в данных на строку 'Unknown':

In [ ]:
data.replace('?', 'Unknown', inplace=True)

In [ ]:
data[data == '?'].any().sum()

Пропусков в данных больше нет

Исходя из нашего EDA удалим ненужные признаки `education` и `fnlwgt`:

In [ ]:
data.drop(['education', 'fnlwgt'], axis=1, inplace=True)

In [ ]:
data.head(3)

Заменим признак `native.country` на новый признак сформированный на его основе - `is_usa`. Новый признак будет иметь только 3 возможных значения 'yes' - человек родом из USA, 'no' - человек родом из любой другой страны кроме USA и 'Unknown' - место рождения неизвестно.

In [ ]:
data['is_usa'] = np.where(
    data['native.country'] == 'United-States', 'yes',
    np.where(data['native.country'] == 'Unknown', 'Unknown', 'no')
)
data.drop('native.country', axis=1, inplace=True)

## 2. Разделение данных

Разделим данные на train и test выборки, train - 80%, test - 20%

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['income']), data['income'], test_size=0.2, random_state=52)
X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

## 3. Кодирование категориальных признаков

Сохраним копии даных до кодирования для catboost, он сам хорошо кодирует их

In [ ]:
X_train.to_csv('data/X_train_raw.csv', index=False)
X_test.to_csv('data/X_test_raw.csv', index=False)

Выделим все категориальные признаки

In [ ]:
cat_cols = X_train.select_dtypes(include='str').columns.tolist()

In [ ]:
print(f"Категориальные признаки: {', '.join(cat_cols)}")

Будем кодировать признаки с помощью one-hot encoding

In [ ]:
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True, dtype=int)

In [ ]:
X_train.head(3)

Сохраним данные после кодирования признаков для дальнейшего использования там где не нужно масштабирование

In [ ]:
X_train.to_csv('data/X_train_encoded.csv', index=False)
X_test.to_csv('data/X_test_encoded.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

## 4. Масштабирование числовых признаков

Некоторые модели (LogReg, SVM, KNN) чувствительны к масштабу признаков, поэтому перед обучением
приведём числовые признаки к единому масштабу через `StandardScaler`

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled =  pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
X_train_scaled.head(3)

Сохраним отмасштабированные данные в файлах для дальнейшего использования

In [ ]:
X_train_scaled.to_csv('data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('data/X_test_scaled.csv', index=False)

Всё, можно переходить к самим моделям